## 선택 · 심화 문제 1-1. 요구조건 기반 baseline 추천기

### 문제 배경

Transformer가 최신이라는 이유만으로 모든 문제에 선택하지 않습니다. 긴 문맥·학습 병렬성·스트리밍 조건을 받아 **첫 비교 실험 후보**와 근거를 반환합니다.

### 시작 코드

```python
cases = [
    {"name": "실시간 센서", "long_context": False, "streaming": True, "parallel_training": False},
    {"name": "긴 문서 분류", "long_context": True, "streaming": False, "parallel_training": True},
    {"name": "짧은 배치 분류", "long_context": False, "streaming": False, "parallel_training": False},
]

def recommend_architecture(case):
    raise NotImplementedError
```

### 수행 요구사항

1. 긴 문맥 또는 병렬 학습이 중요하면 `Transformer`와 해당 근거를 반환하세요.
2. 위 조건이 없고 streaming이면 `RNN/LSTM`을 반환하세요.
3. 어느 쪽도 아니면 `baseline 비교 필요`를 반환하세요.
4. 결과는 `model`, `reasons`, `limitations`를 가진 dictionary여야 합니다.

### 제출 결과

- 세 사례 추천표
- 추천기가 실제 benchmark를 대체하지 못하는 이유
- `심화 문제 1 자동 검증: PASS`

### 자동 검증

```python
results = [recommend_architecture(case) for case in cases]
assert [r["model"] for r in results] == ["RNN/LSTM", "Transformer", "baseline 비교 필요"]
assert all(r["reasons"] and r["limitations"] for r in results)
print("심화 문제 1 자동 검증: PASS")
```

    ```
    
    **상세 해설** · 규칙은 구조의 첫 후보를 정할 뿐입니다. 실제 선택은 동일 데이터·metric·하드웨어에서 정확도, 처리량, p95 latency, peak memory를 비교해야 합니다.
    
    **자주 하는 실수**
    
    - Streaming이면 언제나 RNN이 최고라고 결론 냅니다.
    - 근거 없이 model 이름만 반환합니다.
    - 같은 조건의 benchmark 없이 추천 규칙을 최종 의사결정으로 사용합니다.

---

## 최종 자동 검증 및 제출 체크

세 정답 셀의 PASS를 확인한 뒤 아래 항목을 한 행으로 정리하세요. 숫자만 옮기지 말고 구조를 선택하거나 보류한 이유도 함께 적습니다.

```
1-1 | 사례 | 문맥 길이 | streaming | 후보 구조 | 구조 근거 | 예상 병목 | 확인할 benchmark
```

---

In [3]:
cases = [
    {"name": "실시간 센서", "long_context": False, "streaming": True, "parallel_training": False},
    {"name": "긴 문서 분류", "long_context": True, "streaming": False, "parallel_training": True},
    {"name" : "짧은 배치 분류", "long_context": False, "streaming": False, "parallel_training": False},
]

def recommend_architecture(case):
    reasons = []
    # Transformer가 필요한 조건을 먼저 모아 추천 근거로 그대로 재사용합니다.
    if case["long_context"]:
        reasons.append("먼 위치를 짧은 정보 경로로 연결해야 함")
    if case["parallel_training"]:
        reasons.append("sequence 위치의 병렬 학습이 중요함")
    if reasons:
        model = "Transformer"
    elif case["streaming"]:
        model = "RNN/LSTM"
        reasons.append("도착하는 입력에 이전 상태를 이어 쓰는 baseline이 단순함")
    else:
        model = "baseline 비교 필요"
        reasons.append("현재 조건만으로 구조 우위를 단정할 수 없음")
    return {
        "model": model,
        "reasons": reasons,
        # 교육용 규칙이라는 한계를 항상 결과에 포함합니다.
        "limitations": "정확도·지연·메모리·데이터 규모를 같은 조건에서 측정해야 함",
    }

results = [recommend_architecture(case) for case in cases]
for case, result in zip(cases, results):
    print(case["name"], "->", result["model"], "/", "; ".join(result["reasons"]))

assert [r["model"] for r in results] == ["RNN/LSTM", "Transformer", "baseline 비교 필요"]
assert all(r["reasons"] and r["limitations"] for r in results)
print("심화 문제 1 자동 검증: PASS")

실시간 센서 -> RNN/LSTM / 도착하는 입력에 이전 상태를 이어 쓰는 baseline이 단순함
긴 문서 분류 -> Transformer / 먼 위치를 짧은 정보 경로로 연결해야 함; sequence 위치의 병렬 학습이 중요함
짧은 배치 분류 -> baseline 비교 필요 / 현재 조건만으로 구조 우위를 단정할 수 없음
심화 문제 1 자동 검증: PASS
